In [2]:
import gerar_mapa

print(gerar_mapa.__file__)

C:\Users\roger\OneDrive - cefet-rj.br\Empresas e entrevistas\projeto logistica offshore\ofi\src\gerar_mapa.py


In [3]:
import folium
from folium.plugins import AntPath
import pandas as pd
import geopandas as gpd
import json
from gerar_mapa import BASE, LINK_HOME

print(LINK_HOME)

https://rogerfidelis.github.io/ofi/index.html


In [5]:
import folium
from folium.plugins import AntPath
import pandas as pd
import geopandas as gpd
import json
from gerar_mapa import BASE, LINK_HOME

# =========================
# MAPA BASE
# =========================
SURVEY_STARNAV = BASE / "dados" / "SURVEY_STARNAV.xlsx"


dfs = pd.read_excel(SURVEY_STARNAV, sheet_name=None)

vessels = pd.read_excel(BASE / "dados" / "SURVEY_VESSELS"/"SURVEY_VESSELS.xlsx")

for celula in vessels["starnav"].dropna():
    #print (celula)
    mapa = folium.Map(
        location=[-22.872174, -41.983981],
        zoom_start=9
    )

    # =========================
    # SHAPEFILES (ANP)
    # =========================
    shp_campos = gpd.read_file(
    BASE /
    "shapefiles" /
    "CAMPOS_PRODUCAO_SIRGAS" /
    "CAMPOS_PRODUCAO_SIRGASPolygon.shp"
)

    if shp_campos.crs is None or shp_campos.crs.to_epsg() != 4326:
        shp_campos = shp_campos.to_crs(epsg=4326)

    folium.GeoJson(
        data=json.loads(shp_campos.to_json()),
        name="Campos de Produção (ANP)",
        style_function=lambda x: {
            "fillColor": "yellow",
            "color": "orange",
            "weight": 2,
            "fillOpacity": 0.3
        },
        tooltip=folium.GeoJsonTooltip(
            fields=["NOM_CAMPO"],
            aliases=["Campo:"]
        )
    ).add_to(mapa)

    # =========================
    # EXCEL
    # =========================
    #dfs = pd.read_excel("SURVEY_CBO.xlsx", sheet_name=None)
    #dfs = pd.read_excel("SURVEY_BRAM.xlsx", sheet_name=None)


    pontos = dfs['posicoes'].dropna(subset=["latitude", "longitude"])

    cores = {
        "porto":       "red",
        "navio sonda": "blue",
        "FPSO":        "purple",
        "Semi-Sub/Prod/Perfuração": "green",
        "Semi-Sub/Perfuração":      "orange",
        "Fixa (Habitada)":          "darkred",
        "Semi-Sub/Produção":        "cadetblue",
        "estaleiro": "white"
    }

    # =========================
    # PONTOS FIXOS + ZONA 500 m
    # =========================
    for _, p in pontos.iterrows():
        folium.CircleMarker(
            [p["latitude"], p["longitude"]],
             radius=6,
            color=cores[p["tipo"]],
            fill=True,
            fill_opacity=0.7,
            popup=f'{p["unidade"]} - {p["tipo"]}'
        ).add_to(mapa)

        #folium.Circle(
           # [p["latitude"], p["longitude"]],
          #  radius=500,
         #   color=cores[p["tipo"]],
       #     fill=True,
      #      fill_opacity=0.15
      #  ).add_to(mapa)

    # =========================
    # TRAJETÓRIA GPS
    # =========================
    #nome_embarcacao = "CBO COPACABANA"
    #nome_embarcacao = "BRAM BRAVO"
    #nome_embarcacao = "STARNAV LIBRA"


    #cbo = dfs[nome_embarcacao].dropna(subset=["lat_a", "lon_a"])
    cbo = dfs[celula.upper()].dropna(subset=["lat_a", "lon_a"])
    trajetoria = cbo[["lat_a", "lon_a"]].values.tolist()
    print(trajetoria)

    for _, p in cbo.iterrows():
        folium.CircleMarker(
            [p["lat_a"], p["lon_a"]],
            #[cbo["lat_a"], cbo["lon_a"]],
            radius=3,
            color="blue",
            fill=True,
            popup=f"""
            <b>Hora:</b> {p["hora_reportada"]}<br>
            <b>Data:</b> {p["data_reportada"]}
            <b>Status:</b> {p["status"]}
            """
         ).add_to(mapa)

        AntPath(
            locations=trajetoria,
            color="blue",
            weight=5,
            delay=900,
            dash_array=[5, 10],
            pulse_color="white"
        ).add_to(mapa)

        # =========================
        # INÍCIO E FIM
        # =========================
        inicio = cbo.iloc[0]
        fim = cbo.iloc[-1]

        folium.Marker(
            trajetoria[0],
            icon=folium.Icon(color="green", icon="play"),
            popup=f'INÍCIO: {inicio["hora_reportada"]} - {inicio["data_reportada"]}'
        ).add_to(mapa)

        folium.Marker(
            trajetoria[-1],
            icon=folium.Icon(color="red", icon="stop"),
            popup=f'FIM: {fim["hora_reportada"]} - {fim["data_reportada"]}'
        ).add_to(mapa)

        # =========================
        # SALVAR
        # =========================
        #mapa.save("cbo_copacabana.html")
        #mapa.save("bram_bravo.html")
        #mapa.save("starnav_libra.html")
        ##########################################################################
        
        legenda = """
        <div style="
        position:fixed;
        bottom:25px;
        right:25px;
        
        width:min(280px, 90vw);
        @media (max-width:768px){

            .legenda{

                bottom:15px;

                right:15px;

                left:15px;

                width:auto;

                font-size:12px;

                padding:10px;

            }

        }
        
        background:#0B1F3A;
        border:2px solid #1CA3EC;
        border-radius:10px;
        padding:15px;
        color:white;
        font-family:Arial;
        font-size:14px;
        z-index:9999;
        ">

        <h3 style="margin-top:0;color:#1CA3EC;">
        Legenda
        </h3>

        ● <span style="color:red;">Porto</span><br>
        ● <span style="color:purple;">FPSO</span><br>
        ● <span style="color:blue;">Navio Sonda</span><br>
        ● <span style="color:green;">Semi-Sub/Prod/Perfuração</span><br>
        ● <span style="color:yellow;">Semi-Sub/Perfuração</span><br>
        ● <span style="color:darkred;">Plataforma Fixa</span><br>
        ● <span style="color:white;">Estaleiro</span><br><br>

        

        <span style="color:#00BFFF;">━ ━ ━ ━ </span> Trajetória AIS<br>

        🟢 Início<br>

        🔴 Última posição

        </div>
        """

        mapa.get_root().html.add_child(folium.Element(legenda))
                   
      #######################################################################
       

        mapa.save(BASE/"mapas"/"starnav"/f'{celula.replace(" ", "_")}.html')
        #print(celula.replace(" ", "_") + ".html")
        
    

[[-24.5, -42.62], [-24.7, -42.5], [-24.693716, -42.512161], [-24.688559, -42.506248], [-24.683649, -42.512291], [-24.774258, -42.551895], [-24.689299, -42.507027], [-24.694963, -42.507534], [-24.688181, -42.427677], [-24.688181, -42.427677], [-24.688181, -42.427677], [-24.7139, -42.46146], [-24.7139, -42.46146], [-22.854635, -43.144726], [-22.852852, -43.141617]]
[[-22.85, -43.16], [-22.862665, -43.170013], [-22.867193, -43.210125], [-22.863565, -43.160812], [-23.477707, -42.935436], [-23.779406, -42.49062], [-24.63316, -42.451542], [-24.722353, -42.42128], [-24.726425, -42.417923], [-24.726425, -42.417923], [-24.726425, -42.417923], [-24.726425, -42.417923], [-24.552618, -42.408653], [-24.625423, -42.449715], [-24.762581, -42.353539]]
[[-21.88, -41.01], [-21.99, -40.66], [-22.507704, -39.989899], [-22.953606, -40.726383], [-22.471478, -40.066074], [-21.864775, -41.016354], [-21.864777, -41.01635], [-22.95289, -40.73671], [-22.960093, -40.731628], [-22.95396, -40.727509], [-22.464138, 

In [7]:
import pandas as pd
from gerar_mapa import BASE
from math import radians, sin, cos, sqrt, atan2

arquivo_origem = BASE / "dados" / "SURVEY_CBO.xlsx"
arquivo_saida = BASE / "dados" / "SURVEY_CBO_D1.xlsx"

# Lê todas as abas
planilhas = pd.read_excel(arquivo_origem, sheet_name=None)

# Lê a aba com as posições fixas
posicoes = pd.read_excel(
    arquivo_origem,
    sheet_name="posicoes"
)

# Mantém apenas linhas válidas
posicoes = posicoes.dropna(subset=["latitude", "longitude"])

print(posicoes.head())

print(
    posicoes[
        posicoes["unidade"].str.contains("Anna", case=False, na=False)
    ]
)

with pd.ExcelWriter(arquivo_saida, engine="openpyxl") as writer:
    
    def haversine(lat1, lon1, lat2, lon2):

        R = 6371  # raio da Terra em km

        lat1 = radians(lat1)
        lon1 = radians(lon1)
        lat2 = radians(lat2)
        lon2 = radians(lon2)

        dlat = lat2 - lat1
        dlon = lon2 - lon1

        a = sin(dlat/2)**2 + cos(lat1) * cos(lat2) * sin(dlon/2)**2

        c = 2 * atan2(sqrt(a), sqrt(1-a))

        return R * c


    for nome_aba, df in planilhas.items():
                
        print(f"Processando {nome_aba}")
        
        # Ignora abas sem a coluna data_consulta
        if "data_consulta" not in df.columns:
            print(f"Aba {nome_aba} ignorada.")
            continue

        # Converte para datetime
        df["data_consulta"] = pd.to_datetime(
            df["data_consulta"],
            dayfirst=True
        )
        
        df["data_reportada"] = pd.to_datetime(
        df["data_reportada"],
        dayfirst=True
        )
        
        
        # Calcula diferença
        delta = df["data_consulta"].diff()

        # Cria um novo dataframe somente com os indicadores
        resultado = pd.DataFrame({
            "data_consulta": df["data_consulta"],
            "delta_dia_input": delta,
            "data_reportada": df["data_reportada"]
        })
        
        distancias = [None]
        
        menor_distancia = [None]
        unidade_proxima = [None]
        tipo_unidade = [None]
        campo_ = [None]
        

        for i in range(1, len(df)):

            d = haversine(
                df.loc[i-1, "lat_a"],
                df.loc[i-1, "lon_a"],
                df.loc[i, "lat_a"],
                df.loc[i, "lon_a"]
            )

            #distancias.append(d)
            distancias.append(round(d, 2))
            
            menor = float("inf")
            unidade = None
            tipo = None
            campo= [None]

            lat = df.loc[i, "lat_a"]
            lon = df.loc[i, "lon_a"]

            for _, pos in posicoes.iterrows():

                d = haversine(
                    lat,
                    lon,
                    pos["latitude"],
                    pos["longitude"]
                )

                if d < menor:
                    menor = d
                    unidade = pos["unidade"]
                    tipo = pos["tipo"]
                    campo = pos["campo"]

            menor_distancia.append(round(menor, 3))
            unidade_proxima.append(unidade)
            tipo_unidade.append(tipo)
            campo_.append(campo)

        resultado["distancia_km"] = distancias 
        resultado["delta_horas_input"] = (resultado["delta_dia_input"].dt.total_seconds() / 3600)
        resultado["delta_dia_reportada"] = df["data_reportada"].diff()
        resultado["delta_horas_reportada"] = (resultado["delta_dia_reportada"].dt.total_seconds() / 3600)
        resultado["velocidade_kmh"] = (resultado["distancia_km"]/resultado["delta_horas_reportada"])
        resultado["velocidade_kmh"] = resultado["velocidade_kmh"].round(2)
        resultado["velocidade_knots"] = (resultado["velocidade_kmh"] / 1.852).round(2)
        resultado["status de navegação"]= df["status"]
        
        resultado["menor_distancia_km"] = menor_distancia
        resultado["unidade_proxima"] = unidade_proxima
        resultado["tipo_unidade"] = tipo_unidade
        resultado["campo"]= campo_
        
        # Tempo entre a coleta e a última posição reportada
        resultado["delta_coleta_reportada"] = (
        resultado["data_consulta"] - resultado["data_reportada"]
        )

        # Salva na mesma aba
        resultado.to_excel(
            writer,
            sheet_name=nome_aba,
            index=False
        )

print("Arquivo criado com sucesso!")


                                    unidade   latitude  longitude       tipo  \
0                          estaleiro renave -22.865902 -43.130785  estaleiro   
1                            estaleiro maua -22.877544 -43.129122  estaleiro   
2                      TechnipFMC Flexiveis -21.871379 -41.015639      porto   
3                             NOV Flexiveis -21.870543 -41.018545      porto   
4  Baker Hughes Planta de Fluídos - Niterói -22.882348 -43.118153      porto   

  operador bacia          campo  
0      NaN   NaN      estaleiro  
1      NaN   NaN       staleiro  
2      NaN   NaN      porto açu  
3      NaN   NaN      porto açu  
4      NaN   NaN  porto niteroi  
           unidade  latitude  longitude  tipo operador   bacia campo
31  FPSO ANNA NERY    -22.46     -40.05  FPSO   YINSON  campos   NaN
Processando posicoes
Aba posicoes ignorada.
Processando CBO ALESSANDRA
Processando CBO ALIANCA
Processando CBO ANITA
Processando CBO ARPOADOR
Processando CBO CAMPOS
Processando 

In [8]:
import pandas as pd
from gerar_mapa import BASE

# ==========================
# ARQUIVOS DA FROTA
# ==========================

arquivos = [
    BASE / "dados" / "SURVEY_BRAM_D1.xlsx",
    BASE / "dados" / "SURVEY_CBO_D1.xlsx",
    BASE / "dados" / "SURVEY_STARNAV_D1.xlsx"
]

total_frota = 0

print("=" * 60)

for arquivo in arquivos:

    print(f"\nArquivo: {arquivo.name}")

    planilhas = pd.read_excel(arquivo, sheet_name=None)

    total_empresa = 0

    for nome_aba, df in planilhas.items():

        if "distancia_km" not in df.columns:
            continue

        distancia = df["distancia_km"].sum(skipna=True)

        print(f"{nome_aba:<25} {distancia:10.2f} km")

        total_empresa += distancia

    print("-" * 60)
    print(f"TOTAL {arquivo.stem:<18} {total_empresa:10.2f} km")

    total_frota += total_empresa

print("\n" + "=" * 60)
print(f"TOTAL GERAL DA FROTA: {total_frota:.2f} km")
print("=" * 60)

######################################################################################


#Atualizar os cards da langind page
from pathlib import Path

BASE = Path.cwd().parent      # se estiver executando dentro de src
# ou BASE = Path.cwd()         # se executar na raiz do projeto

script = f"""const ESTATISTICAS = {{
    km: {total_frota},
    empresas: 3,
    psvs: 53,
    campos: 18
}};

document.getElementById("km").dataset.target = ESTATISTICAS.km;
document.getElementById("campos").dataset.target = ESTATISTICAS.campos;
document.getElementById("empresas").dataset.target = ESTATISTICAS.empresas;
document.getElementById("psvs").dataset.target = ESTATISTICAS.psvs;

const counters = document.querySelectorAll(".contador");

counters.forEach(counter => {{

    const target = parseFloat(counter.dataset.target);

    let current = 0;

    const increment = Math.max(1, Math.ceil(target / 80));

    function update(){{

        current += increment;

        if(current >= target){{

            counter.innerText = Math.round(target).toLocaleString("pt-BR");

        }}else{{

            counter.innerText = Math.round(current).toLocaleString("pt-BR");

            requestAnimationFrame(update);

        }}

    }}

    update();

}});
"""

with open(BASE / "script.js", "w", encoding="utf-8") as f:
    f.write(script)

print("script.js atualizado com sucesso.")


Arquivo: SURVEY_BRAM_D1.xlsx
BRAM ATLAS                    397.16 km
BRAM BAHIA                    300.92 km
BRAM BELEM                    573.92 km
BRAM BRASIL                   234.24 km
BRAM BRASILIA                 504.42 km
BRAM BRAVO                    281.31 km
BRAM BREEZE                   612.25 km
BRAM BUCK                     716.10 km
BRAM BUZIOS                   111.96 km
BRAM HERO                     814.34 km
BRAM POWER                    179.47 km
BRAM RIO                      635.76 km
BRAM SPIRIT                   190.33 km
BRAM TITAN                    498.73 km
------------------------------------------------------------
TOTAL SURVEY_BRAM_D1        6050.91 km

Arquivo: SURVEY_CBO_D1.xlsx
CBO ALESSANDRA                114.89 km
CBO ALIANCA                   588.89 km
CBO ANITA                    1340.93 km
CBO ARPOADOR                 2890.65 km
CBO CAMPOS                      1.10 km
CBO CAROLINA                  322.33 km
CBO COPACABANA                901.34 km
C